# Assignment 01 - Vietnam House Price Prediction Intelligent System

## 1. System and Problem Definition

This project builds a supervised regression intelligent system for estimating Vietnam house listing prices.

Input: structured property information. Output: estimated `Price` in billion VND. Formally, the system learns a function `f(x) -> predicted house price`.

This is an educational machine-learning estimate, not an official property appraisal.

## 2. Dataset Source

Dataset: House Price Prediction Dataset Vietnam - 2024  
Kaggle URL: https://www.kaggle.com/datasets/nguyentiennhan/vietnam-housing-dataset-2024  
Local copied path: `data/vietnam_housing_dataset.csv`

Target: `Price`, measured in billion VND.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from house_price_pipeline import run_pipeline

summary = run_pipeline(PROJECT_ROOT, verbose=True)
summary["dataset"]

Experiment 1 done: Linear Regression {'MAE': 1.3096524098412623, 'MSE': 2.802928804623156, 'RMSE': 1.6739770272506143, 'R2': 0.4276094917994073, 'MAPE': 0.28175206958458565}


Experiment 1 done: Decision Tree {'MAE': 1.226400836694591, 'MSE': 3.116083099955625, 'RMSE': 1.7650325879087514, 'R2': 0.3635220959798806, 'MAPE': 0.2497352040855135}


Experiment 1 done: Random Forest {'MAE': 1.051996929762986, 'MSE': 2.098276951065024, 'RMSE': 1.4483278043698398, 'R2': 0.5714305654928852, 'MAPE': 0.218632811062816}


Experiment 1 done: Extra Trees {'MAE': 1.084158044178874, 'MSE': 2.3254624918175133, 'RMSE': 1.5247050830884792, 'R2': 0.5250248007260344, 'MAPE': 0.22295129691885265}


Experiment 1 done: Gradient Boosting {'MAE': 1.2590808159980127, 'MSE': 2.5003952715740825, 'RMSE': 1.581189603565392, 'R2': 0.48933534752304453, 'MAPE': 0.279552948716092}


Experiment 2 done: max_depth 5 {'MAE': 1.3987978549160969, 'MSE': 3.0617833516225774, 'RMSE': 1.7497482203045525, 'R2': 0.37461636125132874, 'MAPE': 0.3126346079315458}


Experiment 2 done: max_depth 10 {'MAE': 1.2066276613316589, 'MSE': 2.3876250568819257, 'RMSE': 1.5450990377911844, 'R2': 0.512309638135176, 'MAPE': 0.25923552792776144}


Experiment 2 done: max_depth 20 {'MAE': 1.0708927323953759, 'MSE': 2.04958300935083, 'RMSE': 1.4313749124957378, 'R2': 0.5813947965180695, 'MAPE': 0.2238473350783027}


Experiment 2 done: max_depth None {'MAE': 1.051996929762986, 'MSE': 2.098276951065024, 'RMSE': 1.4483278043698398, 'R2': 0.5714305654928852, 'MAPE': 0.218632811062816}


{'shape': [30229, 12],
 'columns': ['Address',
  'Area',
  'Frontage',
  'Access Road',
  'House direction',
  'Balcony direction',
  'Floors',
  'Bedrooms',
  'Bathrooms',
  'Legal status',
  'Furniture state',
  'Price'],
 'dtypes': {'Address': 'str',
  'Area': 'float64',
  'Frontage': 'float64',
  'Access Road': 'float64',
  'House direction': 'str',
  'Balcony direction': 'str',
  'Floors': 'float64',
  'Bedrooms': 'float64',
  'Bathrooms': 'float64',
  'Legal status': 'str',
  'Furniture state': 'str',
  'Price': 'float64'},
 'missing_counts': {'Address': 0,
  'Area': 0,
  'Frontage': 11564,
  'Access Road': 13297,
  'House direction': 21239,
  'Balcony direction': 24983,
  'Floors': 3603,
  'Bedrooms': 5162,
  'Bathrooms': 7074,
  'Legal status': 4506,
  'Furniture state': 14119,
  'Price': 0},
 'missing_percent': {'Address': 0.0,
  'Area': 0.0,
  'Frontage': 38.25,
  'Access Road': 43.99,
  'House direction': 70.26,
  'Balcony direction': 82.65,
  'Floors': 11.92,
  'Bedrooms': 

## 3. Initial Inspection

The executed output above reports shape, columns, dtypes, missing counts, missing percentages, duplicates, target range, and numeric descriptive statistics. The raw dataset is validated against the expected 12-column schema before modeling continues.

## 4. Address to Location Feature Engineering

The raw `Address` column has very high cardinality, so the complete raw address is not used directly as an ML feature. Instead, `Province` is extracted from the last comma-separated segment and `District` from the second-last comma-separated segment after Unicode NFC normalization, whitespace trimming, and trailing-dot cleanup.

Example: `Duong Nguyen Van Khoi, Phuong 11, Go Vap, Ho Chi Minh` becomes `District = Go Vap` and `Province = Ho Chi Minh` after the same deterministic parsing rule.

## 5. Feature Representations

Full engineered representation: 12 features obtained by replacing raw `Address` with `Province` and `District`.

Final six-feature representation required for deployment: `Area`, `Floors`, `Bedrooms`, `Bathrooms`, `Province`, `District`.

The six features are chosen because they capture size, structure, property capacity, and location while avoiding several raw attributes with substantial missingness. The experiment later tests whether this simpler representation remains competitive; it is not assumed to be automatically more accurate.

In [2]:
summary["features"]

{'full': ['Area',
  'Frontage',
  'Access Road',
  'House direction',
  'Balcony direction',
  'Floors',
  'Bedrooms',
  'Bathrooms',
  'Legal status',
  'Furniture state',
  'Province',
  'District'],
 'selected': ['Area',
  'Floors',
  'Bedrooms',
  'Bathrooms',
  'Province',
  'District']}

## 6. Exactly Five EDA Distribution Charts

The five EDA distribution charts are:

1. Price Distribution histogram, unit billion VND.
2. Area Distribution histogram.
3. Floors Distribution.
4. Bedrooms Distribution.
5. Top Provinces by Listing Count.

Evaluation charts later are separate and do not count toward these five EDA charts.

In [3]:
summary["eda_charts"]

['figures/01_price_distribution.png',
 'figures/02_area_distribution.png',
 'figures/03_floors_distribution.png',
 'figures/04_bedrooms_distribution.png',
 'figures/05_top_provinces_distribution.png']

## 7. EDA Summary

The target and feature summaries should be interpreted from the executed dataset output. The data contains missingness in several optional property attributes, while `Address`, `Area`, and `Price` are complete. Location is expected to matter because listing counts are concentrated in major provinces and districts, but this is treated as a predictive association, not a causal claim.

## 8. Train/Test Split and Leakage Control

One row-index based split is used for both representations: 80% training and 20% held-out test, `random_state=42`, no stratification because this is regression.

Leakage control:

- Imputers fit only inside training folds.
- Scalers fit only inside training folds.
- OneHotEncoder fits only inside training folds.
- Model selection uses training cross-validation only.
- Hyperparameter experiments use training cross-validation only.
- Representation experiments use training cross-validation only.
- Held-out test is used once after final configuration is locked.

In [4]:
summary["split"]

{'train_rows': 24183, 'test_rows': 6046, 'test_size': 0.2, 'random_state': 42}

## 9. Baseline

The baseline is `DummyRegressor(strategy="median")` wrapped in the same six-feature preprocessing protocol. Metrics are MAE, MSE, RMSE, R2, and MAPE.

In [5]:
summary["baseline"]

{'full': {'MAE Mean': 1.8436555786463351,
  'MAE Std': 0.013964368594094083,
  'MSE Mean': 4.896983120608597,
  'MSE Std': 0.0614007834838133,
  'RMSE Mean': 2.2128694739483796,
  'RMSE Std': 0.013849616457525383,
  'R2 Mean': -0.00014667404832806595,
  'R2 Std': 6.084864285732658e-05,
  'MAPE Mean': 0.447518068035403,
  'MAPE Std': 0.005656411863982985},
 'compact': {'MAE': 1.8436555786463351,
  'MSE': 4.896983120608597,
  'RMSE': 2.2128694739483796,
  'R2': -0.00014667404832806595,
  'MAPE': 0.447518068035403}}

## Traditional Machine Learning Model Understanding

### Linear Regression
A. Input Representation: numeric columns are imputed/scaled and categorical location columns are one-hot encoded into a numeric vector.  
B. Learning Idea: learns a linear weighted relationship between input features and price.  
C. What It Learns: coefficients for each encoded feature and one intercept.  
D. Strengths: simple, fast, interpretable, useful as a linear benchmark.  
E. Weaknesses: cannot naturally capture complex nonlinear interactions among area, district, and structure.  
F. Suitability: helpful baseline for house-price data, but Vietnam listings are likely nonlinear and location-sensitive.

### Decision Tree Regressor
A. Input Representation: the same preprocessed numeric vector.  
B. Learning Idea: recursively creates feature-threshold splits to reduce regression error.  
C. What It Learns: a tree of split rules and leaf predictions.  
D. Strengths: nonlinear, understandable as rules, handles interactions.  
E. Weaknesses: high overfitting risk if unrestricted.  
F. Suitability: can capture local patterns in property data, but may be unstable.

### Random Forest Regressor
A. Input Representation: the same preprocessed vector.  
B. Learning Idea: trains many trees using bagging and random feature sampling.  
C. What It Learns: an ensemble of regression trees whose predictions are averaged.  
D. Strengths: reduces variance compared with one tree and models nonlinear structured data.  
E. Weaknesses: heavier and less interpretable than a single tree.  
F. Suitability: strong candidate for noisy housing listings with nonlinear location and size effects.

### Extra Trees Regressor
A. Input Representation: the same preprocessed vector.  
B. Learning Idea: trains a highly randomized ensemble of trees, including randomized split thresholds.  
C. What It Learns: many randomized tree structures and averaged predictions.  
D. Strengths: can reduce variance and is often strong on tabular data.  
E. Weaknesses: less interpretable and may underfit or over-randomize some patterns.  
F. Suitability: useful comparison against Random Forest for high-cardinality location data.

### Gradient Boosting Regressor
A. Input Representation: the same preprocessed vector.  
B. Learning Idea: sequentially adds trees that correct residual errors from previous trees.  
C. What It Learns: an additive ensemble of weak learners focused on residual improvement.  
D. Strengths: often accurate for nonlinear structured data.  
E. Weaknesses: sensitive to hyperparameters and less parallel due to sequential learning.  
F. Suitability: a strong traditional ML model for property-price regression.

## Experiment 1 - Five Model Comparison

Question: Which traditional regression algorithm provides the strongest performance using the same six-feature representation and preprocessing protocol?

Fixed: `X_train_6/y_train`, six features, preprocessing, CV folds, and metrics. Changed: regression algorithm only. The held-out test is not used.

In [6]:
import pandas as pd
pd.DataFrame(summary["experiment1"]["full_table"])

,Model,MAE Mean,MAE Std,MSE Mean,MSE Std,RMSE Mean,RMSE Std,R2 Mean,R2 Std,MAPE Mean,MAPE Std
0,Linear Regression,1.309652,0.017257,2.802929,0.090146,1.673977,0.027013,0.427609,0.014437,0.281752,0.006526
1,Decision Tree,1.226401,0.015139,3.116083,0.097218,1.765033,0.027259,0.363522,0.019922,0.249735,0.007565
2,Random Forest,1.051997,0.014931,2.098277,0.072581,1.448328,0.024970,0.571431,0.014493,0.218633,0.006139
3,Extra Trees,1.084158,0.015325,2.325462,0.082920,1.524705,0.027146,0.525025,0.016746,0.222951,0.006290
4,Gradient Boosting,1.259081,0.012091,2.500395,0.048356,1.581190,0.015320,0.489335,0.006913,0.279553,0.006106


In [7]:
pd.DataFrame(summary["experiment1"]["compact_table"])

,Model,MAE,MSE,RMSE,R2,MAPE
0,Linear Regression,1.309652,2.802929,1.673977,0.427609,0.281752
1,Decision Tree,1.226401,3.116083,1.765033,0.363522,0.249735
2,Random Forest,1.051997,2.098277,1.448328,0.571431,0.218633
3,Extra Trees,1.084158,2.325462,1.524705,0.525025,0.222951
4,Gradient Boosting,1.259081,2.500395,1.581190,0.489335,0.279553


In [8]:
summary["experiment1"]["best_model"]

'Random Forest'

Evaluation visualizations saved:

- `figures/06_rmse_model_comparison.png`
- `figures/07_r2_model_comparison.png`

## Experiment 2 - Hyperparameter Investigation

Controlled experiment on `RandomForestRegressor(max_depth)` with values `5`, `10`, `20`, and `None`. Fixed: `n_estimators=100`, random state, six features, preprocessing, KFold, and all five metrics.

In [9]:
pd.DataFrame(summary["experiment2"]["table"])

,max_depth,MAE,MSE,RMSE,R2,MAPE
0,5,1.398798,3.061783,1.749748,0.374616,0.312635
1,10,1.206628,2.387625,1.545099,0.512310,0.259236
2,20,1.070893,2.049583,1.431375,0.581395,0.223847
3,None,1.051997,2.098277,1.448328,0.571431,0.218633


In [10]:
summary["experiment2"]["best_rf_max_depth"]

20

The line chart `figures/08_rf_max_depth_vs_rmse.png` shows max_depth versus mean CV RMSE. The candidate model is selected using training CV only by comparing the Experiment 1 winner with the tuned Random Forest.

In [11]:
summary["candidate"]

{'algorithm': 'Random Forest', 'rf_max_depth': 20}

## Experiment 3 - Feature Representation Investigation

This experiment compares the full 12 engineered features against the final six selected features using the same training rows, target, candidate algorithm/configuration, CV folds, preprocessing protocol, and five metrics. The changed variable is feature representation only.

In [12]:
pd.DataFrame(summary["experiment3"]["table"])

,Representation,Feature Count,MAE,MSE,RMSE,R2,MAPE
0,Full Engineered Features,12,1.043913,1.932388,1.389757,0.605347,0.219252
1,Six Selected Features,6,1.070893,2.049583,1.431375,0.581395,0.223847
2,Difference (6 - Full),-6,0.026980,0.117195,0.041618,-0.023953,0.004596


Interpretation should be based on the actual metrics above. If six features are worse, that is a performance/usability trade-off; if they are equal or better, the reduction is empirically supported. Deployment still uses exactly six teacher-required features.

## Scientific Final Configuration and Held-out Test

The final system is locked before viewing the held-out test: six-feature representation, preprocessing inside a pipeline, and the selected algorithm/configuration from training CV. The held-out test is then used once for final scientific evaluation. No tuning occurs after this result.

In [13]:
summary["final_test"]

{'MAE': 1.0788394028207318,
 'MSE': 2.0960686429076176,
 'RMSE': 1.4477805921159523,
 'R2': 0.5701177088445141,
 'MAPE': 0.2239460989835775}

Final evaluation visualizations saved:

- `figures/11_actual_vs_predicted_scatter.png`
- `figures/12_final_residual_distribution.png`
- `figures/13_cv_vs_heldout_rmse.png`

MAE and RMSE are in billion VND. R2 describes explained variance quality. MAPE is an average percentage-type error. The result is not a guaranteed market value.

## Saved Deployment Models

Five six-feature trained pipelines are saved in `models/`:

- `linear_regression.joblib`
- `decision_tree.joblib`
- `random_forest.joblib`
- `extra_trees.joblib`
- `gradient_boosting.joblib`

The scientific final pipeline is also saved as `best_model.joblib`. Streamlit loads these artifacts and never retrains.

In [14]:
summary["demos"]

[{'input': {'Area': 35.0,
   'Floors': 2.0,
   'Bedrooms': 2.0,
   'Bathrooms': 1.0,
   'Province': 'Hà Nội',
   'District': 'Cầu Giấy'},
  'predicted_price_billion': 3.836663914496358},
 {'input': {'Area': 70.0,
   'Floors': 4.0,
   'Bedrooms': 4.0,
   'Bathrooms': 3.0,
   'Province': 'Hồ Chí Minh',
   'District': 'Gò Vấp'},
  'predicted_price_billion': 7.805499987513429},
 {'input': {'Area': 150.0,
   'Floors': 5.0,
   'Bedrooms': 6.0,
   'Bathrooms': 5.0,
   'Province': 'Đà Nẵng',
   'District': 'Cẩm Lệ'},
  'predicted_price_billion': 8.336343607929239}]

## Controlled Experiment Summary

| Experiment | Question | Changed Variable | Fixed Variables | Main Result |
|---|---|---|---|---|
| Experiment 1 | Which algorithm is strongest? | Model algorithm | Six features, preprocessing, CV folds, metrics | Lowest CV RMSE model is selected as the initial winner |
| Experiment 2 | Which RF max_depth is best? | Random Forest max_depth | n_estimators, six features, preprocessing, folds, metrics | Best depth is chosen by lowest CV RMSE |
| Experiment 3 | Do full 12 features improve over six? | Feature representation | Candidate algorithm, train rows, folds, metrics | Reported honestly using all five metrics |


## Reflection

What worked: structured preprocessing, five-model comparison, Province/District engineering, controlled experiments, Streamlit deployment design, and Neo4j Knowledge Graph integration.

Challenges: missing data, noisy high-cardinality addresses, categorical encoding, regression generalization, and deployment packaging.

Limitations: the dataset contains property listings and may not equal final transaction prices; it represents 2024 market conditions; real estate markets change over time; Province/District extraction is simplified; several raw attributes have substantial missing data; the six-feature deployed representation is a simplification; predictions are not official valuations.

## Conclusion

Raw Data -> Data Representation -> EDA -> Feature Engineering -> Train/Test -> Baseline -> Five ML Models -> Experiment 1 -> Experiment 2 -> Experiment 3 -> Final Six-Feature Model -> Held-out Evaluation -> Streamlit -> Neo4j -> Deployment.